In [1]:
import os
import re
import json
import spacy
import numpy as np
import joblib
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

/home/arch/.pyenv/versions/3.11.8/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class AudiologyPipeline:
    def __init__(self, modelDir="./models", templatesDir="./templates", sbertCheckpoint='fine_tuned_audiology_sbert/checkpoint-505'):
        releventStopwords = set(json.load(open(os.path.join(templatesDir, "relevantStopwords.json"), "r"))) 
        self.canonicalSymtoms = json.load(open(os.path.join(templatesDir, "canonicalSymptoms.json"), "r"))
        self.symptomNormalization = json.load(open(os.path.join(templatesDir, "symptomNormalizations.json"), "r"))

        self.nlp = spacy.load("en_core_web_sm")

        for word in releventStopwords:
            self.nlp.vocab[word].is_stop = True

        self.embedder = SentenceTransformer(os.path.join(modelDir, sbertCheckpoint))

        self.clf = joblib.load(os.path.join(modelDir, "audiology_lr_classifier.pkl"))
        self.mlb = joblib.load(os.path.join(modelDir, "audiology_label_binarizer.pkl"))

        self.symptomEmbeddings = self.embedder.encode(self.canonicalSymtoms)

    def _cleanText(self, text):
        # Normalize and clean the input text
        text = text.lower()
        text = re.sub(r'[^a-z\s]', ' ', text)

        for colloquial, standard in self.symptomNormalization.items():
            text = text.replace(colloquial, standard)

        doc = self.nlp(text)
        tokens = [token.text for token in doc if not token.is_stop and token.text.strip()]

        return ' '.join(tokens)

    def _extractSymptoms(self, text, theshold=0.55):
        # Extract symptoms from clean text
        doc = self.nlp(text)
        pahrases = [chunk.text for chunk in doc.noun_chunks if chunk.text.strip()]
        pahrases.extend([token.lemma_ for token in doc if token.pos_ in ["ADJ", "VERB"]])
        pahrases.append(text)

        if not pahrases:
            return []

        pahraseEmbeddings = self.embedder.encode(list(set(pahrases)))
        similarity = cosine_similarity(pahraseEmbeddings, self.symptomEmbeddings)

        extracted = set()
        for i, _ in enumerate(pahrases):
            bestIndex = np.argmax(similarity[i])
            if similarity[i][bestIndex] > theshold:
                extracted.add(self.canonicalSymtoms[bestIndex])

        return list(extracted)

    def _predictConditions(self, text, threshold=0.4):
        # Encodes the text and predicts the highest-probability conditions

        vecInput = self.embedder.encode([text])
        probabilities = self.clf.predict_proba(vecInput)[0]

        validIndices = np.where(probabilities > threshold)[0]
        if len(validIndices)>0:
            sortValidIncdices = validIndices[np.argsort(-probabilities[validIndices])]
            return self.mlb.classes_[sortValidIncdices].tolist()

        return "Unknown Condition (Clinical evaluation recommended)"
    
    def process(self, text):
        # Execute the full pipeline

        cleanText = self._cleanText(text)
        symptoms = self._extractSymptoms(cleanText)
        conditions = self._predictConditions(cleanText)

        return {
            "input_text": text,
            "clean_text": cleanText,
            "extracted_symptoms": symptoms,
            "predicted_conditions": conditions
        }


In [9]:
engine = AudiologyPipeline()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2232.33it/s]


In [10]:
user = "doc, im dealing with everything sounds muffled, plus my er is burning pain."

In [11]:
result = engine.process(user)

In [12]:
result

{'input_text': 'doc, im dealing with everything sounds muffled, plus my er is burning pain.',
 'clean_text': 'doc m dealing muffled_hearing_hearing plus er ear_pain',
 'extracted_symptoms': ['hearing loss'],
 'predicted_conditions': ['Noise-Induced Hearing Loss',
  'Otitis Media (Middle Ear Infection)']}